<a href="https://colab.research.google.com/github/Mertcanyucedag/NesneTespitAlgoritmalarindaMarjGenisletmesininDogrulukOranlariUzerineEtkisi/blob/Furkan/YGA20_Model_Egitimi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# YOLOv8 ve gerekli kütüphanelerin kurulumu
!pip install ultralytics albumentations wandb opencv-python-headless

In [ ]:
import albumentations as A
import cv2

# 1. Sis Efekti (Blur) [cite: 121, 122]
def apply_fog(image):
    aug = A.Compose([A.Blur(blur_limit=7, p=1.0)])
    return aug(image=image)['image']

# 2. Yağmur Efekti (Noise/Gürültü) [cite: 123, 124]
def apply_rain(image):
    aug = A.Compose([A.GaussNoise(var_limit=(10.0, 50.0), p=1.0)])
    return aug(image=image)['image']

# 3. Gece/Düşük Işık (Brightness) [cite: 125, 126]
def apply_night(image):
    aug = A.Compose([A.RandomBrightnessContrast(brightness_limit=(-0.5, -0.5), contrast_limit=0, p=1.0)])
    return aug(image=image)['image']

In [ ]:
import wandb
from ultralytics import YOLO

# 1. W&B Projesini Başlatma [cite: 104]
# Proje ismini ve deneyi takip edeceğimiz ana başlığı belirliyoruz
wandb.init(project="YGA-20-BoundingBox-Analysis", name="Baseline-Model-YOLOv8")

# 2. YOLOv8 Modelini Yükleme [cite: 92]
# Mimari tasarımın için 'yolov8n.pt' (nano) modelini temel alıyoruz.
# Hem hızlı eğitilir hem de karşılaştırma yapmak için idealdir.
model = YOLO('yolov8n.pt')

print("Model ve Takip Sistemi Hazır!")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: furkanalkan1104 (furkanalkan1104-a) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Model ve Takip Sistemi Hazır!


In [ ]:

import yaml

# Bu dosya modelin hangi sınıfları tanıyacağını ve veri yolunu belirler
data_config = {
    'path': '/content/datasets', # Verilerin ineceği ana klasör
    'train': 'train/images',     # Eğitim görselleri
    'val': 'val/images',         # Doğrulama (validation) görselleri
    'names': {
        0: 'nesne_adi'           # Tespit edilecek nesne (Örn: Araba, İnsan vb.)
    }
}

with open('data.yaml', 'w') as f:
    yaml.dump(data_config, f)